<a href="https://colab.research.google.com/github/nedokormysh/OpenEdu_HSE_INTRML/blob/week8_random_forest/HSE_Task_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Загрузите данные load_wine из sklearn.datasets. Из обучающей части исключите объекты класса 2. Обучите случайный лес, задав только гиперпараметры `n_estimators=100` и `random_state=0`. Оцените важность признаков. Укажите название двух наиболее важных признаков.

In [1]:
from sklearn.datasets import load_wine
data = load_wine()

In [2]:
import pandas as pd

In [6]:
X = pd.DataFrame(data['data'], columns=data['feature_names'])
y = pd.DataFrame(data['target'])

In [7]:
X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [12]:
df = pd.concat([X, y], axis=1)
df.rename(columns = {0: 'target'}, inplace = True)
df = df[df['target'] != 2]

In [13]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [14]:
df1 = df.copy()
y1 = df1['target']
X1 = df1.drop('target', axis = 1)

In [15]:
from sklearn.ensemble import RandomForestClassifier

rfc1 = RandomForestClassifier(n_estimators = 100, random_state = 0)
rfc1.fit(X1, y1)

RandomForestClassifier(random_state=0)

In [16]:
pd.DataFrame({
    'feature_name': X1.columns,
    'importance': rfc1.feature_importances_
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

,feature_name,importance
0,proline,0.298094
1,alcohol,0.216588
2,color_intensity,0.191830
3,flavanoids,0.096270
4,magnesium,0.065010
5,alcalinity_of_ash,0.029383
6,total_phenols,0.028665
7,malic_acid,0.022743
8,ash,0.014751
9,od280/od315_of_diluted_wines,0.010124


2. Загрузите данные load_wine из sklearn.datasets. Из обучающей части исключите объекты класса 2. Отмасштабируйте признаки, используя класс StandardScaler с гиперпараметрами по умолчанию. Обучите случайный лес, задав только гиперпараметры  `n_estimators = 100` и `random_state=0`. Оцените важность признаков. Укажите название двух наиболее важных признаков.

In [17]:
df2 = df.copy()
y2 = df2['target']
X2 = df2.drop('target', axis = 1)

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X2 = scaler.fit_transform(X2)

In [19]:
rfc2 = RandomForestClassifier(n_estimators = 100, random_state = 0)
rfc2.fit(X2, y2)

RandomForestClassifier(random_state=0)

In [22]:
pd.DataFrame({
    'feature_name': X1.columns,
    'importance': rfc2.feature_importances_}).sort_values(by='importance', ascending=False).reset_index(drop=True)

,feature_name,importance
0,proline,0.298094
1,alcohol,0.216588
2,color_intensity,0.191830
3,flavanoids,0.096270
4,magnesium,0.065010
5,alcalinity_of_ash,0.029383
6,total_phenols,0.028665
7,malic_acid,0.022743
8,ash,0.014751
9,od280/od315_of_diluted_wines,0.010124


Ниже приведена неполная реализация класса Bagging который имеет методы `fit` для обучения бэггинга над `DecisionTreeRegressor` и метод `predict` для предсказания. Допишите необходимый код, чтобы реализовать бэггинг.

используемы переменные в коде:
- `self.n_estimators`, `n_estimators` - число используемых деревьев
- `self.regressors` - список объектов класса `DecisionTreeRegressor`, к которым уже был применён метод `fit`
Данный список необъодимо заполнить в методе `fit` и использовать для предсказания в методе `predict`
- `ind`-  выбранные индексы объектов при бутстрапе

при создании объекта класса `DecisionTreeRegressor` зафиксируйте  
`random_state=0`

In [39]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
class Bagging():
  def __init__(self, n_estimators=10):
    self.n_estimators = n_estimators
    self.regressors = []
  def fit(self, x_train, y_train):
    for i in range(self.n_estimators):
      np.random.seed(i)
      ind = np.random.choice(np.arange(x_train.shape[0]), size = x_train.shape[0])
      x_ = x_train.iloc[ind, :]
      y_ = y_train.iloc[ind]
      dtr = DecisionTreeRegressor(random_state=0)
      dtr.fit(x_, y_)
      self.regressors.append(dtr)
  def predict(self, x_test):
    answs = 0
    for regr in self.regressors:
      answs += regr.predict(x_test)
    return answs / self.n_estimators

Загрузите данные приложенные к заданию

In [26]:
# import pandas as pd
# from google.colab import files
# uploder = files.upload()

In [30]:
Data = pd.read_csv('/content/data (1).csv', header = None)

In [31]:
Data.head()

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,0.510632,-1.313935,-0.114209,-0.143679,2.107414,1.507354,-0.315861,-1.030258,0.644616,0.531194,...,-0.892854,-0.699436,0.442937,1.134091,-0.355533,-1.355772,0.363396,1.294192,0.750299,-37.192781
1,0.538866,0.387687,-1.137239,-1.092431,0.038607,1.142312,-0.321904,-0.237227,-1.451496,-0.385336,...,-0.056560,1.248607,-0.346863,-0.973055,1.042978,-0.404316,0.028007,-2.658630,-2.001785,-220.464123
2,-0.068804,0.776603,-1.611731,-1.239927,0.712057,2.075371,-1.462034,1.151911,1.269848,0.725593,...,1.989607,-0.021897,-0.372160,-0.870986,1.192751,-0.062865,0.209232,-0.350511,0.409299,-325.380249
3,-0.884799,-0.620779,0.841720,-1.129511,-0.966720,-1.577867,0.058787,0.223962,-1.005101,0.113334,...,0.214109,-1.083825,-0.476753,-0.904678,-0.637636,-0.741085,-0.126770,-1.063897,-0.003012,-126.861161
4,-0.701277,0.614840,0.434146,-0.126249,-0.591696,0.500336,1.018807,0.848046,-0.817596,-0.060330,...,-0.257656,-1.565868,0.049286,0.239042,-1.047043,0.191627,0.071640,0.766967,1.570351,92.835567


Положим матрицу объекты-признаки в переменную `X`, а ответы в переменную `y`

In [32]:
X, y = Data.iloc[:, :100], Data.iloc[:, 100]


Положим первые 6000 объектов в обучающую часть, остальные объекты в тестовую часть

In [33]:
x_train, y_train = X[:6000], y[:6000]
x_test, y_test = X[6000:], y[6000:]

3. Обучите бэггинг на 1 дереве. Оцените качество по метрике MSE на тестовой части. Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

In [41]:
bg = Bagging(n_estimators=1)
bg.fit(x_train, y_train)

In [36]:
from sklearn.metrics import mean_squared_error

In [44]:
round(mean_squared_error(bg.predict(x_test), y_test) / 1000, 1)

33.9

In [48]:
round(mean_squared_error(bg.predict(x_test), y_test) / 1000)

34

4. Обучите бэггинг на 5 деревьях. Оцените качество по метрике MSE на тестовой части. Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

In [45]:
bg5 = Bagging(n_estimators=5)
bg5.fit(x_train, y_train)

In [47]:
round(mean_squared_error(bg5.predict(x_test), y_test) / 1000)

15

5. Обучите бэггинг на 100 деревьях. Оцените качество по метрике MSE на тестовой части. . Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

In [50]:
bg100 = Bagging(n_estimators=100)
bg100.fit(x_train, y_train)

In [51]:
round(mean_squared_error(bg100.predict(x_test), y_test) / 1000)

11

6. Обучите на этих же данных случайный лес, используйте гиперпараметр `n_estimators = 1`, зафиксируйте  
`random_state=0`. Оцените качество по метрике MSE на тестовой части. . Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

In [52]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators = 1, random_state = 0)
rf.fit(x_train, y_train)

RandomForestRegressor(n_estimators=1, random_state=0)

In [53]:
round(mean_squared_error(rf.predict(x_test), y_test) / 1000)

35

7. Обучите на этих же данных случайный лес, используйте гиперпараметр `n_estimators = 1`, зафиксируйте  
`random_state=0`. Оцените качество по метрике MSE на тестовой части. . Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

8. Обучите на этих же данных случайный лес, используйте гиперпараметр `n_estimators = 5`, зафиксируйте  
`random_state=0`. Оцените качество по метрике MSE на тестовой части. . Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

In [54]:
rf5 = RandomForestRegressor(n_estimators = 5, random_state = 0)
rf5.fit(x_train, y_train)

RandomForestRegressor(n_estimators=5, random_state=0)

In [55]:
round(mean_squared_error(rf5.predict(x_test), y_test) / 1000)

15

9. Обучите на этих же данных случайный лес, используйте гиперпараметр `n_estimators = 100`, зафиксируйте  
`random_state=0`. Оцените качество по метрике MSE на тестовой части. . Ответ разделите на 1000 и округлите до целой части по математичестким правилам округления.

In [56]:
rf100 = RandomForestRegressor(n_estimators = 100, random_state = 0)
rf100.fit(x_train, y_train)

RandomForestRegressor(random_state=0)

In [57]:
round(mean_squared_error(rf100.predict(x_test), y_test) / 1000)

11

10. Изучите документацию и разберитесь как посчитать Out-of-bag ошибку в RandomForestRegressor. Обучите RandomForestRegressor с гиперпараметром n_estimators=100 на обучающей части, зафиксируйте  
`random_state=0`. Найдите Out-of-bag ошибку алгоритма. Ответ округлите до сотых.

In [58]:
rf_oob = RandomForestRegressor(n_estimators = 100, random_state = 0, oob_score=True)
rf_oob.fit(x_train, y_train)

RandomForestRegressor(oob_score=True, random_state=0)

In [59]:
round(rf_oob.oob_score_, 2)

0.75